# Dusha Fine-Tune — WavLM Base Full Fine-Tune (Kaggle)

Trains `xbgoose/wavlm-base-speech-emotion-recognition-russian-dusha-finetuned`  
on crowd-aggregated Dusha labels (Dawid-Skene or Majority Vote).

**Input в Kaggle:**
- `dusha-datasetcrowd` — аудиофайлы (wavs/)
- твой датасет с агрегированным TSV (aggregated_*.tsv)

**Выбор разметки:** поменяй `AGGREGATED_TSV` ниже.

## 1. GPU check

In [ ]:
import subprocess, sys, os
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available")

## 2. Install dependencies

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torchaudio', 'transformers>=4.40', 'datasets>=2.18', 'peft>=0.10',
    'scikit-learn', 'matplotlib', 'seaborn', 'soundfile', 'pyyaml', 'tqdm',
], check=True)
print('Done.')

## 3. Clone repo

In [ ]:
REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

## 4. Config

Укажи имя датасета с агрегированными TSV — обучится по очереди для каждой агрегации.

In [ ]:
import warnings, logging, pathlib
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

from src.config import load_config

# ── укажи имена датасетов ─────────────────────────────────────────────────────
AGGREGATED_DATASET = '<твой-датасет-train>'   # агрегированные train TSV
TEST_DATASET       = '<твой-датасет-test>'    # агрегированные test TSV
# ─────────────────────────────────────────────────────────────────────────────

AGG_ROOT  = pathlib.Path(f'/kaggle/input/{AGGREGATED_DATASET}')
TEST_ROOT = pathlib.Path(f'/kaggle/input/{TEST_DATASET}')

AGGREGATIONS = {
    'majority': 'aggregated_majority.tsv',
    'ds_0.85':  'aggregated_ds_0.85.tsv',
    'ds_0.9':   'aggregated_ds_0.9.tsv',
    'ds_0.95':  'aggregated_ds_0.95.tsv',
    'ds_0.98':  'aggregated_ds_0.98.tsv',
}
TRAIN_TSVS = {tag: AGG_ROOT  / fn for tag, fn in AGGREGATIONS.items()}
TEST_TSVS  = {tag: TEST_ROOT / fn for tag, fn in AGGREGATIONS.items()}

# автодетект базовой папки датасета (с/без лишнего вложения)
_base = next(
    (p for p in [
        '/kaggle/input/dusha-datasetcrowd',
        '/kaggle/input/dusha-datasetcrowd/dusha-datasetcrowd',
    ] if (pathlib.Path(p) / 'crowd_train' / 'wavs').exists()),
    '/kaggle/input/dusha-datasetcrowd',
)
TRAIN_AUDIO_DIR = f'{_base}/crowd_train'
TEST_AUDIO_DIR  = f'{_base}/crowd_test'

print(f'TRAIN_AUDIO_DIR : {TRAIN_AUDIO_DIR}')
print(f'  wavs/ exists  : {(pathlib.Path(TRAIN_AUDIO_DIR) / "wavs").exists()}')
print(f'TEST_AUDIO_DIR  : {TEST_AUDIO_DIR}')
print(f'  wavs/ exists  : {(pathlib.Path(TEST_AUDIO_DIR) / "wavs").exists()}')

import torch, pandas as pd
n_gpus = torch.cuda.device_count()
print(f'\nGPUs: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}  '
          f'{torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

print('\nTrain TSVs:')
for tag, path in TRAIN_TSVS.items():
    n = len(pd.read_csv(path, sep='\t')) if path.exists() else 0
    print(f'  {tag:12s}  {"✓" if path.exists() else "✗"}  {n:6,} rows')
print('\nTest TSVs:')
for tag, path in TEST_TSVS.items():
    n = len(pd.read_csv(path, sep='\t')) if path.exists() else 0
    print(f'  {tag:12s}  {"✓" if path.exists() else "✗"}  {n:6,} rows')

## 5. Train all 5 models

In [ ]:
import gc, random
import numpy as np
import torch.nn as nn
from transformers import AutoFeatureExtractor

from src.dataset import get_dusha_dataloaders
from src.models import build_model
from src.trainer import Trainer

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_dp = torch.cuda.device_count() >= 2
print(f'Device: {device}  |  DataParallel: {use_dp} ({torch.cuda.device_count()} GPUs)')

for tag, tsv_path in TRAIN_TSVS.items():
    if not tsv_path.exists():
        print(f'\n[{tag}] train TSV not found — skip')
        continue

    print(f'\n{"="*60}')
    print(f'  Training: {tag}  ({tsv_path.name})')
    print(f'{"="*60}')

    for _var in ['train_loader', 'dev_loader', 'trainer', 'model']:
        if _var in dir(): del _var
    gc.collect(); torch.cuda.empty_cache()

    config = load_config('configs/wavlm_base_dusha_full.yaml')
    config.aggregated_tsv = str(tsv_path)
    config.audio_dir      = TRAIN_AUDIO_DIR
    config.num_workers    = 0
    config.run_name       = f'wavlm_base_dusha_{tag}'
    config.output_dir     = f'outputs/wavlm_base_dusha_{tag}'
    set_seed(config.seed)

    processor = AutoFeatureExtractor.from_pretrained(
        config.processor_name or config.model_name)
    train_loader, dev_loader, _ = get_dusha_dataloaders(config, processor)
    print(f'Train: {len(train_loader.dataset):,} | Dev: {len(dev_loader.dataset):,}', flush=True)

    model = build_model(config)
    if use_dp:
        model = nn.DataParallel(model)
        print(f'DataParallel: {torch.cuda.device_count()} GPUs', flush=True)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)', flush=True)

    trainer = Trainer(model, config, train_loader, dev_loader, dev_loader, device)
    trainer.fit()

    out = pathlib.Path(config.output_dir)
    print(f'\nCheckpoints in {out}:')
    for f in sorted(out.iterdir()):
        print(f'  {f.name:45s}  {f.stat().st_size/1e6:6.1f} MB')

print('\n\nAll training done!')

## 6. Cross-evaluation matrix

Каждая из 5 моделей тестируется на каждом из 5 тестовых TSV.  
Строки = модель (обучена на), Столбцы = тест (размечен как).  
Значение = weighted accuracy.

In [ ]:
import torch, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm
from src.dataset import get_dusha_test_dataloader
from src.models import build_model

tags = list(AGGREGATIONS.keys())
matrix = np.full((len(tags), len(tags)), np.nan)

for i, model_tag in enumerate(tags):
    ckpt_path = pathlib.Path(f'outputs/wavlm_base_dusha_{model_tag}/best_model.pt')
    if not ckpt_path.exists():
        print(f'[{model_tag}] checkpoint not found — skip row')
        continue

    config = load_config('configs/wavlm_base_dusha_full.yaml')
    config.audio_dir   = TEST_AUDIO_DIR
    config.num_workers = 0
    processor = AutoFeatureExtractor.from_pretrained(
        config.processor_name or config.model_name)

    model = build_model(config)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval().to(device)

    for j, test_tag in enumerate(tags):
        test_tsv = TEST_TSVS[test_tag]
        if not test_tsv.exists():
            continue

        loader = get_dusha_test_dataloader(
            str(test_tsv), TEST_AUDIO_DIR, processor, config)

        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in tqdm(loader,
                              desc=f'  model={model_tag} test={test_tag}',
                              leave=False):
                inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
                preds = model(**inputs).argmax(dim=-1).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(batch['labels'].numpy())

        wacc = balanced_accuracy_score(
            np.concatenate(all_labels), np.concatenate(all_preds))
        matrix[i, j] = wacc
        print(f'  model={model_tag:10s}  test={test_tag:10s}  wacc={wacc:.4f}')

    del model; gc.collect(); torch.cuda.empty_cache()

matrix_df = pd.DataFrame(matrix, index=tags, columns=tags)
matrix_df.index.name = 'trained on \\ tested on'
matrix_df.to_csv('outputs/cross_eval_matrix.tsv', sep='\t', float_format='%.4f')
print('\nMatrix saved → outputs/cross_eval_matrix.tsv')
print()
print(matrix_df.to_string(float_format='{:.4f}'.format))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    matrix_df.astype(float),
    annot=True, fmt='.3f', cmap='YlGn',
    vmin=0, vmax=1,
    xticklabels=tags, yticklabels=tags,
    ax=ax,
)
ax.set_xlabel('Tested on')
ax.set_ylabel('Trained on')
ax.set_title('Cross-evaluation: Weighted Accuracy')
plt.tight_layout()
plt.savefig('outputs/cross_eval_matrix.png', dpi=150)
plt.show()